# Grammar Scoring Engine

## End-to-end project

This notebook demonstrates a complete audio grammar-scoring pipeline. The bundled dataset contains 100 training and 100 test WAV files generated as a self-contained demo. **The demo labels are proxy labels, not official competition labels.**

For the real competition, replace the bundled data with the official 769/216 samples.

### Pipeline
Audio → transcript → linguistic features + TF-IDF → regression → 0–5 score

In [ ]:
!pip -q install numpy pandas scipy scikit-learn matplotlib seaborn librosa soundfile tqdm joblib textstat spacy faster-whisper sentence-transformers
!python -m spacy download en_core_web_sm -q

In [ ]:
from pathlib import Path
import re, os, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import spacy, textstat
from scipy.stats import pearsonr
from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.metrics import mean_squared_error

BASE=Path('.')
train=pd.read_csv(BASE/'train.csv')
test=pd.read_csv(BASE/'test.csv')
print('Train:',train.shape,'Test:',test.shape)
display(train.head())

## 1. Dataset exploration

In [ ]:
print(train['grammar_score'].describe())
plt.figure(figsize=(7,4))
sns.countplot(data=train,x='grammar_score')
plt.title('Training Grammar Score Distribution')
plt.show()

print('Audio files:',len(list((BASE/'audio').glob('*.wav'))))

## 2. Use provided transcripts when available

The demo CSV includes the reference transcript so it runs quickly and deterministically. For the official competition CSV, if `transcript` is absent, the production pipeline should create it with Faster-Whisper.

In [ ]:
nlp=spacy.load('en_core_web_sm',disable=['ner'])
FILLERS={'um','uh','erm','hmm','like','actually','basically','sort of','kind of','i mean'}

def div(a,b): return float(a)/float(b) if b else 0.0

def linguistic_features(text):
    text=str(text or '')
    doc=nlp(text)
    words=[t for t in doc if t.is_alpha]
    sents=list(doc.sents)
    counts={}
    for t in words: counts[t.pos_]=counts.get(t.pos_,0)+1
    toks=[t.text.lower() for t in words]
    unique=len(set(toks))
    repetitions=sum(toks[i]==toks[i-1] for i in range(1,len(toks)))
    lower=text.lower()
    fillers=sum(len(re.findall(r'\b'+re.escape(f)+r'\b',lower)) for f in FILLERS)
    depths=[]
    for t in words:
        d=0; cur=t
        while cur.head!=cur and d<30: d+=1; cur=cur.head
        depths.append(d)
    incomplete=0
    for s in sents:
        ws=[t for t in s if t.is_alpha]
        if len(ws)>=3 and not any(t.pos_ in {'VERB','AUX'} for t in ws): incomplete+=1
    return [len(text),len(words),len(sents),div(len(words),len(sents)),unique,div(unique,len(words)),div(counts.get('NOUN',0),len(words)),div(counts.get('VERB',0),len(words)),div(counts.get('ADJ',0),len(words)),div(counts.get('ADV',0),len(words)),div(counts.get('PRON',0),len(words)),div(counts.get('CCONJ',0)+counts.get('SCONJ',0),len(words)),np.mean(depths) if depths else 0,max(depths) if depths else 0,div(repetitions,len(words)),div(fillers,len(words)),div(incomplete,len(sents)),textstat.flesch_reading_ease(text) if words else 0,textstat.flesch_kincaid_grade(text) if words else 0]

Xeng=np.asarray([linguistic_features(x) for x in train.transcript.fillna('')])
Xeng_test=np.asarray([linguistic_features(x) for x in test.transcript.fillna('')])
print('Engineered feature shape:',Xeng.shape)

## 3. TF-IDF + engineered feature model

In [ ]:
y=train.grammar_score.astype(float).values
vectorizer=TfidfVectorizer(ngram_range=(1,2),min_df=1,max_features=10000,sublinear_tf=True)
Xtext=vectorizer.fit_transform(train.transcript.fillna(''))
Xtest_text=vectorizer.transform(test.transcript.fillna(''))
scaler=StandardScaler()
Xeng=scaler.fit_transform(Xeng)
Xeng_test=scaler.transform(Xeng_test)
X=hstack([Xtext,csr_matrix(Xeng)]).tocsr()
Xtest=hstack([Xtest_text,csr_matrix(Xeng_test)]).tocsr()
print(X.shape)

## 4. 5-fold cross-validation

Leaderboard metrics are RMSE and Pearson correlation.

In [ ]:
def rmse(a,b): return mean_squared_error(a,np.clip(b,0,5))**0.5
kf=KFold(n_splits=5,shuffle=True,random_state=42)
model=Ridge(alpha=10.0)
oof=cross_val_predict(model,X,y,cv=kf)
print(f'CV RMSE: {rmse(y,oof):.4f}')
print(f'CV Pearson: {pearsonr(y,np.clip(oof,0,5))[0]:.4f}')

In [ ]:
plt.figure(figsize=(7,6))
sns.scatterplot(x=y,y=np.clip(oof,0,5))
plt.plot([0,5],[0,5],'--')
plt.xlabel('Actual')
plt.ylabel('OOF Predicted')
plt.title('Out-of-Fold Actual vs Predicted')
plt.show()

## 5. Final training and compulsory training RMSE

In [ ]:
model.fit(X,y)
train_pred=np.clip(model.predict(X),0,5)
TRAIN_RMSE=rmse(y,train_pred)
TRAIN_PEARSON=pearsonr(y,train_pred)[0]
print(f'REQUIRED TRAINING RMSE: {TRAIN_RMSE:.6f}')
print(f'Training Pearson: {TRAIN_PEARSON:.6f}')

## 6. Generate test predictions and submission

In [ ]:
test_pred=np.clip(model.predict(Xtest),0,5)
submission=pd.read_csv(BASE/'sample_submission.csv')
submission['grammar_score']=test_pred
submission.to_csv(BASE/'submission.csv',index=False)
display(submission.head())
print('Saved submission.csv')

## 7. Error/residual analysis

In [ ]:
res=y-train_pred
plt.figure(figsize=(8,4))
sns.histplot(res,bins=20,kde=True)
plt.axvline(0,ls='--')
plt.title('Training Residuals')
plt.xlabel('Residual')
plt.show()

## 8. Save model bundle

In [ ]:
bundle={'model':model,'vectorizer':vectorizer,'scaler':scaler,'feature_fn_version':1}
joblib.dump(bundle,BASE/'model.joblib')
print('Saved model.joblib')

## Conclusion

The project benchmarks a text-centric grammar regression pipeline and reports the required training RMSE plus cross-validation RMSE/Pearson. The bundled demo is intended to verify that the entire workflow runs end-to-end. For the actual competition, use the official labelled audio and avoid treating the demo proxy scores as ground truth.